# 🏆 Data Catalog & Architecture: Gold Layer

--- 

## 📌 Executive Summary & Architecture Overview

The **Gold Layer** represents the business-level data representation within our Enterprise Data Platform (Medallion Architecture standard). It is structured using **dimensional modeling** (Star Schema / Snowflake Schema) to support optimized analytical queries, reporting, dashboards, and advanced data science use cases.

### Key Architectural Principles:
- **Surrogate Keys:** Isolated surrogate keys (`customer_key`, `product_key`) decouple downstream reporting from changes in operational system identifiers.
- **Normalized Facts & Dimensions:** Optimized for fast joins and aggregations across analytical tools (Power BI, Tableau, SQL engines).
- **High Integrity:** Data strictly cleansed, validated, and enriched during transition from the Silver layer.

--- 

## 📐 Star Schema Relationship Model

```
     +------------------------+             +------------------------+
     |   gold.dim_customers   |             |    gold.dim_products   |
     +------------------------+             +------------------------+
     | PK  customer_key       |<---+   +--->| PK  product_key        |
     |     customer_id        |    |   |    |     product_id         |
     |     first_name         |    |   |    |     product_name       |
     |     country            |    |   |    |     category           |
     +------------------------+    |   |    +------------------------+
                                   |   |
                        +----------+---+----------+
                        |     gold.fact_sales     |
                        +-------------------------+
                        | order_number (Degenerate|
                        | FK  product_key         |
                        | FK  customer_key        |
                        |     order_date          |
                        |     sales_amount        |
                        |     quantity            |
                        +-------------------------+
```

--- 

## 📘 1. `gold.dim_customers`

- **Entity Type:** Dimension Table
- **Grain:** One row per unique customer record.
- **Purpose:** Stores customer master records enriched with geographic and demographic classifications for cohort and dynamic segmentation analysis.

### Column Specifications

| Column Name | Data Type | Primary / Foreign Key | Description & Business Rules |
| :--- | :--- | :--- | :--- |
| `customer_key` | **INT** | **PK (Surrogate)** | Unique surrogate key generated in the Gold layer to track distinct customer instances. |
| `customer_id` | **INT** | Natural Key | Primary numeric key assigned in the source CRM operational database. |
| `customer_number` | **NVARCHAR(50)** | Business Key | Alphanumeric account code (e.g., `CUST-8492`) used across operational systems. |
| `first_name` | **NVARCHAR(50)** | - | Customer's primary given name, standardized to Title-Case. |
| `last_name` | **NVARCHAR(50)** | - | Customer's family name or surname. |
| `country` | **NVARCHAR(50)** | - | Standardized country of residence (e.g., 'Australia', 'United States'). |
| `marital_status` | **NVARCHAR(50)** | - | Categorical status ('Married', 'Single', 'Unknown'). |
| `gender` | **NVARCHAR(50)** | - | Normalized gender designation ('Male', 'Female', 'Non-Binary', 'n/a'). |
| `birthdate` | **DATE** | - | Customer date of birth formatted as `YYYY-MM-DD`. Used for demographic age cohorting. |
| `create_date` | **DATE** | - | System timestamp indicating when the customer account was created in the platform. |

--- 

## 📦 2. `gold.dim_products`

- **Entity Type:** Dimension Table
- **Grain:** One row per distinct product SKU or variant.
- **Purpose:** Houses product taxonomy, product series, maintenance requirements, and baseline cost attributes.

### Column Specifications

| Column Name | Data Type | Primary / Foreign Key | Description & Business Rules |
| :--- | :--- | :--- | :--- |
| `product_key` | **INT** | **PK (Surrogate)** | Unique surrogate key identifying each product entry in the catalog. |
| `product_id` | **INT** | Natural Key | Internal product system identifier from the inventory catalog system. |
| `product_number` | **NVARCHAR(50)** | Business Key | Structured alphanumeric code / SKU representing the exact inventory item. |
| `product_name` | **NVARCHAR(50)** | - | Full commercial name of the product including line, color, and size specs. |
| `category_id` | **NVARCHAR(50)** | Foreign Key | Identifier for high-level product taxonomy node. |
| `category` | **NVARCHAR(50)** | - | Broad product grouping (e.g., 'Bikes', 'Components', 'Accessories'). |
| `subcategory` | **NVARCHAR(50)** | - | Granular sub-classification under the parent category (e.g., 'Mountain Bikes'). |
| `maintenance_required` | **NVARCHAR(50)** | - | Flag indicating whether mandatory servicing is required ('Yes', 'No'). |
| `cost` | **INT** | - | Unit production or procurement cost measured in base monetary units. |
| `product_line` | **NVARCHAR(50)** | - | Target market series or product collection (e.g., 'Road', 'Mountain', 'Touring'). |
| `start_date` | **DATE** | - | Date when this product specification became active for commercial distribution. |

--- 

## 📊 3. `gold.fact_sales`

- **Entity Type:** Transactional Fact Table
- **Grain:** One row per order line-item.
- **Purpose:** Records sales transactions, revenue performance, shipment timeframes, and order quantities.

### Column Specifications

| Column Name | Data Type | Primary / Foreign Key | Description & Business Rules |
| :--- | :--- | :--- | :--- |
| `order_number` | **NVARCHAR(50)** | Degenerate Key | Unique order header identifier (e.g., `SO54496`). Groups multiple line-items. |
| `product_key` | **INT** | **FK** | Foreign key linking to `gold.dim_products.product_key`. |
| `customer_key` | **INT** | **FK** | Foreign key linking to `gold.dim_customers.customer_key`. |
| `order_date` | **DATE** | FK (Date Dim) | Transaction placement date (`YYYY-MM-DD`). Primary timeline metric. |
| `shipping_date` | **DATE** | FK (Date Dim) | Fulfillment shipping departure date. Used to compute delivery SLAs. |
| `due_date` | **DATE** | FK (Date Dim) | Payment deadline. Used for accounts receivable analysis. |
| `sales_amount` | **INT** | Measure (Sum) | Total revenue generated for this line item (`sales_amount = price * quantity`). |
| `quantity` | **INT** | Measure (Sum) | Quantity of items sold in this specific order line. |
| `price` | **INT** | Measure (Avg) | Effective selling price per unit applied to the line item. |

--- 

## 💡 Analytical Query Examples

Below are standard analytical queries that can be executed directly against this Gold Layer structure.

In [ ]:
# Example 1: Revenue & Volume Summary by Category & Subcategory
query_category_performance = """
SELECT 
    p.category,
    p.subcategory,
    COUNT(DISTINCT f.order_number) AS total_orders,
    SUM(f.quantity) AS total_units_sold,
    SUM(f.sales_amount) AS total_gross_revenue,
    AVG(f.price) AS average_unit_price
FROM gold.fact_sales f
JOIN gold.dim_products p ON f.product_key = p.product_key
GROUP BY p.category, p.subcategory
ORDER BY total_gross_revenue DESC;
"""
print("Query 1 Prepared: Revenue by Product Category")

In [ ]:
# Example 2: Customer Demographics & Top Revenue Regions
query_customer_demographics = """
SELECT 
    c.country,
    c.gender,
    COUNT(DISTINCT c.customer_key) AS total_customers,
    SUM(f.sales_amount) AS total_revenue,
    ROUND(SUM(f.sales_amount) / COUNT(DISTINCT c.customer_key), 2) AS revenue_per_customer
FROM gold.fact_sales f
JOIN gold.dim_customers c ON f.customer_key = c.customer_key
GROUP BY c.country, c.gender
ORDER BY total_revenue DESC;
"""
print("Query 2 Prepared: Regional Revenue Breakdown")